In [6]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

#setup project root and paths
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"

#path to put results
output_dir = project_root/"results"/"graphs_DAGSLAM"
output_dir.mkdir(parents=True,exist_ok=True)

#test it works
print("CP roots:" , cp_root)
print("Output directory:", output_dir)

CP roots: /dcs/23/u2200504/thesis/recidivism-causal/data/raw/CausalPitfallsData
Output directory: /dcs/23/u2200504/thesis/recidivism-causal/results/graphs_DAGSLAM


In [47]:
#add dagslam implementation to system path
dagslam_path = project_root/"code"/"dagslam"/"DAGSLAM Causal Bayesian Network Structure Learning of Mixed Type Data"/"_dagslam"
sys.path.append(str(dagslam_path))

#uses DAGSLAM implementation code authored by Yuanyuan Zhao. 
#https://github.com/yuanyuan-zhao-pku/DAGSLAM/blob/main/DAGSLAM%20Causal%20Bayesian%20Network%20Structure%20Learning%20of%20Mixed%20Type%20Data/_dagslam/DAGSLAM.py
#DAGSLAM is an extension of the NOTEARS algorithm developed by Xun Zheng, et al.
import importlib
import DAGSLAM 
importlib.reload(DAGSLAM)

from DAGSLAM import dagslam

#m_vec gives the total number of categories for each multinomial variable
#helper function to infer loss types and generate m_vec for each column
def infer_type(df):
    loss_type=[]
    m_vec=[]

    for col in df.columns:
        x=df[col].dropna()
        unique_vals=x.unique()
        num_unique = len(unique_vals)

        #infer variable type by combination of dtype and cardinality
        if np.issubdtype(x.dtype, np.number):
            #numeric datatypes
            if num_unique == 2 and set(unique_vals).issubset({0,1}): #binary numeric variable
                loss_type.append("logistic")
                m_vec.append(1) # 1 "category"
            else: #continuous 
                loss_type.append("gauss")
                m_vec.append(1)
        else: #non-numeric
            if num_unique == 2:
                #binary categorical 
                loss_type.append("logistic")
                m_vec.append(1)
            else:
                #multi-class categorical
                loss_type.append("multi-logistic")
                m_vec.append(num_unique)

    return loss_type, m_vec

def run_dagslam(df, lambda1=0.03, max_iter=100, w_threshold=0.25): #changed lambda 1 0.03 ->0.1 w_threshold 0.25->0.3
    df_clean = df.dropna().copy()
    loss_type, m_vec =infer_type(df_clean)
    X=df_clean.values
    
    W_est = dagslam(X, loss_type=loss_type, m_vec=m_vec, lambda1=lambda1,max_iter=max_iter, w_threshold=w_threshold)
    return W_est

#draw graphs based on DAGSLAM weighted adjacency matrix
def draw_graph(W, output_path, node_labels=None,threshold=0):
    n = W.shape[0] # number of nodes
    G = nx.DiGraph() # initialise empty directed graph
    
    #initialise default node labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]

    #add nodes to digraph
    for i, name in enumerate(node_labels):
        G.add_node(i, label=name)

    #add edges for surviving weights (thresholding done during the dagslam phase
    for i in range(n):
        for j in range(n): # for each possible edge 
            w=W[i,j]
            if abs(w)>threshold:
                G.add_edge(i, j, weight=w)

    plt.figure(figsize=(6,6))
    pos = nx.spring_layout(G, seed=0)
    nx.draw(G, pos, with_labels=True, labels={i: node_labels[i] for i in range(n)},
            node_size=800, font_size=8, arrowsize=10)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

In [48]:
#recurse over all csv datasets and run DAGSLAM
csv_files = sorted(cp_root.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV files")

for csv_path in csv_files:
    rel = csv_path.relative_to(cp_root)
    print(f"Processing: {rel}")

    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("  Skipped (empty file)")
            continue

        # Run DAGSLAM on this dataset
        W_est = run_dagslam(df)

        #Output schema scenario__file__dagslam.png
        parts = rel.parts           
        scenario = parts[0] if len(parts) > 1 else "root"
        name_no_ext = csv_path.stem

        out_name = f"{scenario}__{name_no_ext}__dagslam.png"
        out_path = output_dir / out_name

        # Save PNG
        draw_graph(W_est, out_path, node_labels=list(df.columns))
        print(f"  Saved graph to {out_path.relative_to(project_root)}")

    except Exception as e:
        print(f"  ERROR on {rel}: {e}")

Found 75 CSV files
Processing: berkson_paradox/admission_bias.csv
iter:0
rho:1.0
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
h:0.0
######################################################
  Saved graph to results/graphs_DAGSLAM/berkson_paradox__admission_bias__dagslam.png
Processing: berkson_paradox/hiring_bias.csv
iter:0
rho:1.0
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.3862943611198904
loss=1.38629436111

/tmp/dcs-tmp.u2200504/ipykernel_740464/3451402987.py:78: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/dcs-tmp.u2200504/ipykernel_740464/3451402987.py:78: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1832320968472478
loss=1.1830967565077315
loss=1.1830947310573212
loss=1.183094388326723
h:3.480870404359848e-06
######################################################
rho:10000000.0
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.1830515669936839
loss=1.183051566993

/tmp/dcs-tmp.u2200504/ipykernel_740464/3451402987.py:78: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/dcs-tmp.u2200504/ipykernel_740464/3451402987.py:78: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/dcs-tmp.u2200504/ipykernel_740464/3451402987.py:78: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved graph to results/graphs_DAGSLAM/berkson_paradox__movie_success_bias__dagslam.png
Processing: casual_effect/device_failure_data.csv
iter:0
rho:1.0
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1017324.3379617037
loss=1007276.8507611175
loss=1007267.003036272
loss=1007266.7845597798
loss=1007266.4181043891
loss=1007266.0300447727
loss=1007265.2379064964
loss=1007260.866783478
loss=1007246.776083099
loss=1007185.6248030893
loss=1006891.8399333246
loss=1006128.3760008046
loss=1004647.3980108136
loss=999219.8932416352
loss=992698.9699173283
loss=96

...

/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/scipy/linalg/_matfuncs.py:389: RuntimeWarning: overflow encountered in matmul
  eAw = eAw @ eAw
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/scipy/linalg/_matfuncs.py:389: RuntimeWarning: invalid value encountered in matmul
  eAw = eAw @ eAw


loss=1.2235954289112144e+17
loss=1.2346594062644912e+17
loss=1.240482958376726e+17
loss=1.2661519144801446e+17
loss=1.278977062759982e+17
loss=1.3121003819536243e+17
loss=1.3271899837049544e+17
loss=1.34177960417769e+17
loss=1.357751248997295e+17
loss=1.4122689538338698e+17
loss=1.3619722389062848e+17
loss=1.3634979501536736e+17
loss=1.3626606610133816e+17
loss=1.3632508338861523e+17
loss=1.3669598103483542e+17
loss=1.4010939739549411e+17
loss=1.4105920789322134e+17
loss=1.4098951908906942e+17
loss=1.4155968808512288e+17
loss=1.4247831342925176e+17
loss=1.4515295233382976e+17
loss=1.4958713960835005e+17
loss=1.4945130447880714e+17
loss=1.492305414130327e+17
loss=1.488031900001699e+17
loss=1.4788912820033245e+17
loss=1.4496757540045568e+17
loss=1.4255759115531315e+17
loss=1.411845487917033e+17
loss=1.4132649101453048e+17
loss=1.4142806526176824e+17
loss=1.4171936847907928e+17
loss=1.4266228118137894e+17
loss=1.4464520916627414e+17
loss=1.455302970254657e+17
loss=1.4600702635067894e+17
l

/tmp/dcs-tmp.u2200504/ipykernel_740464/3451402987.py:78: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/dcs/23/u2200504/thesis/recidivism-causal/code/dagslam/DAGSLAM Causal Bayesian Network Structure Learning of Mixed Type Data/_dagslam/DAGSLAM.py:104: RuntimeWarning: invalid value encountered in logaddexp
  * (np.logaddexp(0, M[..., j]) - X[..., j] * M[..., j])
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)


loss=1096.7523033172456
loss=1096.6810438435905
loss=1096.2998677123692
loss=1096.2384737435214
loss=1095.9742426231544
loss=1094.832113280338
loss=1095.801656543705
loss=1095.5696134675843
loss=1095.3317873690246
loss=1094.8798292636125
loss=1095.4360712261755
loss=1094.6880381178967
loss=1094.7955234992476
loss=1094.678786544813
loss=1094.371146635881
loss=1093.9909119091985
loss=1094.368541625814
loss=1094.0315708078017
loss=1094.1497405401096
loss=1093.6082704010516
loss=1093.6690121011536
loss=1237.1134744342282
loss=1093.750868288571
loss=1093.6726122001842
loss=1093.6671902943183
loss=1093.4160433801035
loss=1093.262615476958
loss=1092.788737442029
loss=1093.1131475891493
loss=1092.3124389487107
loss=1092.9799315621879
loss=1092.5206895222275
loss=1092.8688593100235
loss=1092.4490132192295
loss=1092.3769941766918
loss=1092.14586889434
loss=1092.020714779344
loss=1092.124613368426
loss=1092.1059174269942
loss=1091.9900319993403
loss=1091.8127173077262
loss=1091.895033784678
loss=

/dcs/23/u2200504/thesis/recidivism-causal/code/dagslam/DAGSLAM Causal Bayesian Network Structure Learning of Mixed Type Data/_dagslam/DAGSLAM.py:104: RuntimeWarning: invalid value encountered in logaddexp
  * (np.logaddexp(0, M[..., j]) - X[..., j] * M[..., j])
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/dcs/23/u2200504/thesis/recidivism-causal/code/dagslam/DAGSLAM Causal Bayesian Network Structure Learning of Mixed Type Data/_dagslam/DAGSLAM.py:104: RuntimeWarning: invalid value encountered in logaddexp
  * (np.logaddexp(0, M[..., j]) - X[..., j] * M[..., j])
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
